# Linear Probes: Evaluating Latent Space Quality

## Background

A **linear probe** is the standard way to evaluate self-supervised representations (JEPA, etc.):

1. Freeze the trained encoder weights.
2. Attach a single linear layer on top of the frozen features.
3. Train only that linear layer to predict a known ground-truth quantity (e.g., object position).

If this simple classifier succeeds, the information already exists in the encoder's representation in a linearly accessible form. The probe is too simple to learn complex features on its own. It can only read out structure the encoder has already organized. A nonlinear probe (multi-layer network) could compute new features from raw activations, which would not tell us much about what the encoder learned.

## Why this matters

For a world model to support downstream tasks (planning, control, prediction), its latent space must encode meaningful scene properties, not just memorize pixel patterns. Linear probes test whether structured, accessible information about the physical world is present.

## Experimental design

**Properties probed:**
- Object position (center of mass x, y)
- Object velocity (motion vector dx, dy between consecutive frames)
- Motion direction (velocity angle discretized into 8 compass bins)

**Models compared:**
1. Pixel prediction (Laplacian loss, epoch 30)
2. JEPA (latent prediction, epoch 30, if checkpoint exists)
3. Random (same architecture, untrained weights) as baseline

The random baseline shows what performance comes from architecture inductive bias alone. A trained model must beat this to demonstrate meaningful learned representations.

**Dataset:** Moving MNIST (2,000 sequences, 20 frames each, 64x64 grayscale)

**Reference:** Standard linear evaluation protocol per I-JEPA (Assran et al., CVPR 2023) and V-JEPA (Bardes et al., 2024).

## Cell 1: Imports and Setup

Libraries, project paths, compute device (GPU/CPU), output directories for figures and analysis.

Dependencies: PyTorch (model loading, tensors), scikit-learn (R2, accuracy, t-SNE), matplotlib (figures), FluidWorldModelV2 (encoder architecture).

If any path shows `exists=False`, run the training pipeline first. Figures go to `paper/figures/`.

In [ ]:
import os, sys, time, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from pathlib import Path
import matplotlib
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, accuracy_score

# â”€â”€ Project paths â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
PROJECT = Path(r"C:\DEV\Workspace\active\coding\_AI RESEARCH\FluidWorld")
sys.path.insert(0, str(PROJECT))

FIGURES_DIR = PROJECT / "paper" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

ANALYSIS_DIR = PROJECT / "experiments" / "analysis"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = PROJECT / "data" / "mnist_test_seq.npy"
CKPT_PIXEL = PROJECT / "checkpoints" / "moving_mnist" / "model_epoch_30.pt"
CKPT_JEPA  = PROJECT / "checkpoints" / "moving_mnist_jepa" / "model_epoch_30.pt"

from fluidworld.core.world_model_v2 import FluidWorldModelV2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Figures dir: {FIGURES_DIR} (exists={FIGURES_DIR.exists()})")
print(f"Data path:   {DATA_PATH} (exists={DATA_PATH.exists()})")
print(f"Pixel ckpt:  {CKPT_PIXEL} (exists={CKPT_PIXEL.exists()})")
print(f"JEPA ckpt:   {CKPT_JEPA} (exists={CKPT_JEPA.exists()})")

## Cell 2: Load Data and Extract Ground-Truth Labels

Loads Moving MNIST and computes three ground-truth label sets directly from pixel values. These serve as the answer key for the linear probes.

### Label computation

1. **Position (center of mass):** Pixel intensities as weights, compute weighted average of coordinates. Gives (x, y) in [0, 63].

2. **Velocity (finite difference):** `velocity_t = position_{t+1} - position_t`, producing a 2D vector (dx, dy) in pixels/frame. 20 frames yield 19 velocity vectors per sequence.

3. **Direction class (8-way):** `angle = atan2(dy, dx)` discretized into 8 equal 45-degree bins (E, NE, N, NW, W, SW, S, SE).

### Config
- `N_SEQ = 2000` (of 10,000 available)
- `SEED = 42`

**Output:**
- Data: `(2000, 20, 64, 64)` raw, `(2000, 20, 1, 64, 64)` 5D
- Position: `(2000, 20, 2)`, mean near (32, 32)
- Velocity: `(2000, 19, 2)`
- Direction: `(2000, 19)`, roughly uniform across 8 classes (~12.5% each)

These labels come from pixel statistics only. If the encoder's latent space is well-structured, a single linear layer should recover these quantities from the 128-dim feature vector.

In [ ]:
# â”€â”€ Configuration â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
N_SEQ = 2000       # number of sequences to use (out of 10,000)
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# â”€â”€ Load Moving MNIST â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
raw = np.load(str(DATA_PATH))                    # (20, 10000, 64, 64)
raw = raw.transpose(1, 0, 2, 3)                  # (10000, 20, 64, 64)
raw = raw[:N_SEQ].astype(np.float32) / 255.0     # normalize to [0, 1]
data = torch.from_numpy(raw)                      # (N_SEQ, 20, 64, 64)
data_5d = data.unsqueeze(2)                       # (N_SEQ, 20, 1, 64, 64)
print(f"Data shape: {data.shape}  (N_seq, T, H, W)")
print(f"Data 5D:    {data_5d.shape}  (N_seq, T, C, H, W)")

# â”€â”€ Compute center of mass â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def compute_com(frames):
    """
    Compute center of mass for each frame.
    Args:
        frames: (N, T, 1, H, W) tensor, pixel values in [0, 1]
    Returns:
        com: (N, T, 2) tensor â€” (com_x, com_y) coordinates
    """
    B, T, C, H, W = frames.shape
    y_coords = torch.arange(H, dtype=torch.float32).view(1, 1, 1, H, 1)
    x_coords = torch.arange(W, dtype=torch.float32).view(1, 1, 1, 1, W)
    total_mass = frames.sum(dim=(2, 3, 4)) + 1e-8
    com_y = (frames * y_coords).sum(dim=(2, 3, 4)) / total_mass
    com_x = (frames * x_coords).sum(dim=(2, 3, 4)) / total_mass
    return torch.stack([com_x, com_y], dim=-1)  # (B, T, 2)

positions = compute_com(data_5d)  # (N_SEQ, 20, 2)
print(f"Positions shape: {positions.shape}")
print(f"  Mean position: x={positions[:,:,0].mean():.1f}, y={positions[:,:,1].mean():.1f}")

# â”€â”€ Compute velocity (frame-to-frame displacement) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
velocities = positions[:, 1:] - positions[:, :-1]  # (N_SEQ, 19, 2)
print(f"Velocities shape: {velocities.shape}")

# â”€â”€ Compute direction classes (8-way) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def velocity_to_direction_class(vel):
    """
    Discretize velocity angle into 8 compass directions.
    0=E, 1=NE, 2=N, 3=NW, 4=W, 5=SW, 6=S, 7=SE
    Args:
        vel: (N, T, 2) tensor â€” (dx, dy)
    Returns:
        classes: (N, T) LongTensor â€” direction class [0..7]
    """
    angles = torch.atan2(vel[..., 1], vel[..., 0])  # radians, (-pi, pi)
    # Shift so bin 0 centers on East (angle=0)
    angles = (angles + 2 * math.pi) % (2 * math.pi)
    bin_size = 2 * math.pi / 8
    classes = ((angles + bin_size / 2) / bin_size).long() % 8
    return classes

direction_classes = velocity_to_direction_class(velocities)  # (N_SEQ, 19)
print(f"Direction classes shape: {direction_classes.shape}")
direction_names = ['E', 'NE', 'N', 'NW', 'W', 'SW', 'S', 'SE']
unique, counts = torch.unique(direction_classes, return_counts=True)
print("Direction distribution:")
for u, c in zip(unique.tolist(), counts.tolist()):
    print(f"  {direction_names[u]:>3s}: {c:>6d} ({100*c/direction_classes.numel():.1f}%)")

## Cell 3: Load Models and Extract Frozen Features

Instantiates FluidWorldModelV2 three times with identical configs, loads different weights, then extracts frozen latent features from all 2,000 sequences.

### Models
1. **Pixel model**: Laplacian pixel-prediction checkpoint (epoch 30). Trained to predict future frames in pixel space.
2. **JEPA model**: JEPA checkpoint (epoch 30), if it exists. Trained to predict future latent states.
3. **Random model**: Untrained weights. Any feature structure comes purely from architecture inductive bias (conv shapes, normalization), not learning.

### Feature extraction
For each model:
1. Freeze all weights (`requires_grad_(False)`).
2. Encode each frame via `model.encode(x)`, producing spatial feature maps `(batch, 128, 16, 16)`.
3. Global average pool over the 16x16 spatial dims to get a single 128-dim vector per frame. This discards spatial position within the feature map (the probe must recover position from feature values, not grid location).

Result: `(2000, 20, 128)` tensor per model. Missing checkpoints trigger a warning and skip.

In [ ]:
def build_model(device):
    """Construct a FluidWorldModelV2 with the standard Moving MNIST config."""
    return FluidWorldModelV2(
        in_channels=1,
        d_model=128,
        stimulus_dim=1,
        n_encoder_layers=3,
        max_steps_encoder=6,
        belief_spatial_hw=16,
        n_belief_evolve=3,
        loss_type="bce",
        use_fatigue=False,
        use_inhibition=True,
        use_memory_pump=True,
        use_hebbian=True,
        use_deltanet=True,
        use_titans=True,
    ).to(device)


def load_checkpoint(model, ckpt_path, device):
    """Load weights from a checkpoint file."""
    ckpt = torch.load(str(ckpt_path), map_location=device, weights_only=False)
    if isinstance(ckpt, dict):
        sd = ckpt.get("model_state_dict", ckpt.get("state_dict", ckpt))
    else:
        sd = ckpt
    model.load_state_dict(sd, strict=False)
    return model


@torch.no_grad()
def extract_features(model, data_tensor, batch_size=64):
    """
    Extract frozen encoder features for all frames.
    
    Args:
        model: FluidWorldModelV2 (eval mode, frozen)
        data_tensor: (N, T, 1, H, W) float tensor
        batch_size: batch size for encoding
    Returns:
        features: (N, T, 128) tensor â€” globally pooled latent vectors
    """
    model.eval()
    all_features = []
    B_total = data_tensor.shape[0]
    T = data_tensor.shape[1]
    
    n_batches = (B_total + batch_size - 1) // batch_size
    for bi_idx, bi in enumerate(range(0, B_total, batch_size)):
        batch = data_tensor[bi:bi+batch_size].to(device)
        B = batch.shape[0]
        feats_t = []
        for t in range(T):
            enc_out = model.encode(batch[:, t])  # {"features": (B, 128, 16, 16), ...}
            z = enc_out["features"]              # (B, 128, 16, 16)
            z_pooled = z.mean(dim=(2, 3))        # (B, 128)
            feats_t.append(z_pooled.cpu())
        feats_t = torch.stack(feats_t, dim=1)    # (B, T, 128)
        all_features.append(feats_t)
        if (bi_idx + 1) % 5 == 0 or bi_idx == n_batches - 1:
            print(f"  Batch {bi_idx+1}/{n_batches} done")
    
    return torch.cat(all_features, dim=0)  # (N, T, 128)


# â”€â”€ Extract features from each model â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
features = {}  # model_name -> (N, T, 128) tensor

# 1) Pixel prediction model (Laplacian, epoch 30)
print("=" * 60)
print("Loading Pixel model (Laplacian, epoch 30)...")
model_pixel = build_model(device)
load_checkpoint(model_pixel, CKPT_PIXEL, device)
model_pixel.requires_grad_(False)
model_pixel.eval()
t0 = time.time()
features["Pixel"] = extract_features(model_pixel, data_5d)
print(f"  Pixel features: {features['Pixel'].shape}  ({time.time()-t0:.1f}s)")
del model_pixel
torch.cuda.empty_cache() if device.type == "cuda" else None

# 2) JEPA model (epoch 30) â€” skip if checkpoint doesn't exist
if CKPT_JEPA.exists():
    print("=" * 60)
    print("Loading JEPA model (epoch 30)...")
    model_jepa = build_model(device)
    load_checkpoint(model_jepa, CKPT_JEPA, device)
    model_jepa.requires_grad_(False)
    model_jepa.eval()
    t0 = time.time()
    features["JEPA"] = extract_features(model_jepa, data_5d)
    print(f"  JEPA features: {features['JEPA'].shape}  ({time.time()-t0:.1f}s)")
    del model_jepa
    torch.cuda.empty_cache() if device.type == "cuda" else None
else:
    print("=" * 60)
    print(f"JEPA checkpoint not found at {CKPT_JEPA}")
    print("  Skipping JEPA â€” run JEPA training first, then re-run this notebook.")

# 3) Random encoder (untrained weights)
print("=" * 60)
print("Extracting features from Random (untrained) encoder...")
model_random = build_model(device)
model_random.requires_grad_(False)
model_random.eval()
t0 = time.time()
features["Random"] = extract_features(model_random, data_5d)
print(f"  Random features: {features['Random'].shape}  ({time.time()-t0:.1f}s)")
del model_random
torch.cuda.empty_cache() if device.type == "cuda" else None

print("=" * 60)
print(f"Feature sets extracted: {list(features.keys())}")

## Cell 4: Train Linear Probes

Core evaluation. For each encoder (Pixel, JEPA, Random), trains three separate linear probes: a single `nn.Linear(128, output_dim)` with no hidden layers or nonlinearities.

### The three probes

1. **Position probe** (regression, output_dim=2): Predicts center-of-mass (x, y) from a single frame's 128-dim features. Loss: MSE. Metric: R2.

2. **Velocity probe** (regression, output_dim=2): Predicts velocity (dx, dy) at time t. Harder than position since the encoder sees one frame but must implicitly capture motion.

3. **Direction probe** (8-class, output_dim=8): Predicts discretized motion direction. Loss: cross-entropy. Metric: top-1 accuracy (chance = 12.5%).

### Training config
- 80/20 train/val split, shuffled
- Adam, lr=1e-3, 50 epochs, batch size 512
- 50 epochs is plenty for a single linear layer with convex loss

### Why a single linear layer?

The probe computes `output = W @ features + b`. It cannot learn nonlinear features, so any information it recovers was linearly encoded by the frozen encoder.

**Expected results:**

| Metric | Trained models | Random |
|--------|---------------|--------|
| Position R2 | 0.8+ | ~0.0 |
| Velocity R2 | 0.3+ | ~0.0 |
| Direction acc | 0.3+ | ~0.125 |

R2 interpretation: 1.0 = perfect, 0.7-0.99 = strong, 0.3-0.7 = moderate, 0.0 = no better than mean prediction, negative = worse than mean. The gap between trained and Random matters more than absolute values.

In [ ]:
class LinearProbe(nn.Module):
    """A single linear layer â€” the simplest possible probe."""
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim)
    
    def forward(self, x):
        return self.linear(x)


def train_regression_probe(feats, targets, n_epochs=50, lr=1e-3, val_frac=0.2):
    """
    Train a linear probe for regression (MSE loss, RÂ² metric).
    
    Args:
        feats:   (N_samples, 128) tensor
        targets: (N_samples, out_dim) tensor
    Returns:
        r2: RÂ² on validation set
        probe: trained LinearProbe
    """
    N = feats.shape[0]
    n_val = int(N * val_frac)
    n_train = N - n_val
    
    # Shuffle
    perm = torch.randperm(N)
    feats, targets = feats[perm], targets[perm]
    
    X_train, X_val = feats[:n_train], feats[n_train:]
    Y_train, Y_val = targets[:n_train], targets[n_train:]
    
    out_dim = targets.shape[1]
    probe = LinearProbe(128, out_dim).to(device)
    optimizer = torch.optim.Adam(probe.parameters(), lr=lr)
    
    train_ds = TensorDataset(X_train, Y_train)
    train_dl = DataLoader(train_ds, batch_size=512, shuffle=True)
    
    for epoch in range(n_epochs):
        probe.train()
        epoch_loss = 0.0
        for xb, yb in train_dl:
            xb, yb = xb.to(device), yb.to(device)
            pred = probe(xb)
            loss = F.mse_loss(pred, yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * xb.shape[0]
        epoch_loss /= n_train
    
    # Evaluate on validation set
    probe.eval()
    with torch.no_grad():
        preds = probe(X_val.to(device)).cpu().numpy()
        truth = Y_val.numpy()
    r2 = r2_score(truth, preds, multioutput='uniform_average')
    return r2, probe


def train_classification_probe(feats, targets, n_classes=8, n_epochs=50, lr=1e-3, val_frac=0.2):
    """
    Train a linear probe for classification (CE loss, accuracy metric).
    
    Args:
        feats:   (N_samples, 128) tensor
        targets: (N_samples,) LongTensor
    Returns:
        accuracy: accuracy on validation set
        probe: trained LinearProbe
    """
    N = feats.shape[0]
    n_val = int(N * val_frac)
    n_train = N - n_val
    
    perm = torch.randperm(N)
    feats, targets = feats[perm], targets[perm]
    
    X_train, X_val = feats[:n_train], feats[n_train:]
    Y_train, Y_val = targets[:n_train], targets[n_train:]
    
    probe = LinearProbe(128, n_classes).to(device)
    optimizer = torch.optim.Adam(probe.parameters(), lr=lr)
    
    train_ds = TensorDataset(X_train, Y_train)
    train_dl = DataLoader(train_ds, batch_size=512, shuffle=True)
    
    for epoch in range(n_epochs):
        probe.train()
        for xb, yb in train_dl:
            xb, yb = xb.to(device), yb.to(device)
            logits = probe(xb)
            loss = F.cross_entropy(logits, yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    
    # Evaluate on validation set
    probe.eval()
    with torch.no_grad():
        logits = probe(X_val.to(device)).cpu()
        preds = logits.argmax(dim=1).numpy()
        truth = Y_val.numpy()
    acc = accuracy_score(truth, preds)
    return acc, probe


# â”€â”€ Prepare flat feature/label arrays â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Position probe: use all 20 timesteps
# Velocity/direction probe: use timesteps 0..18 (velocity is t+1 - t, so 19 pairs)

results = {}  # model_name -> {"pos_r2", "vel_r2", "dir_acc"}

for model_name, feat_tensor in features.items():
    print(f"\n{'='*60}")
    print(f"Training probes for: {model_name}")
    print(f"{'='*60}")
    
    # feat_tensor: (N_SEQ, 20, 128)
    N, T, D = feat_tensor.shape
    
    # --- Probe 1: Position regression ---
    print("\n  [Probe 1] Position prediction (RÂ²)...")
    pos_feats = feat_tensor.reshape(N * T, D)           # (N*20, 128)
    pos_targets = positions.reshape(N * T, 2)            # (N*20, 2)
    pos_r2, _ = train_regression_probe(pos_feats, pos_targets)
    print(f"    Position RÂ² = {pos_r2:.4f}")
    
    # --- Probe 2: Velocity regression ---
    print("  [Probe 2] Velocity prediction (RÂ²)...")
    # Use features at time t to predict velocity (t -> t+1)
    vel_feats = feat_tensor[:, :-1].reshape(N * (T-1), D)    # (N*19, 128)
    vel_targets = velocities.reshape(N * (T-1), 2)            # (N*19, 2)
    vel_r2, _ = train_regression_probe(vel_feats, vel_targets)
    print(f"    Velocity RÂ² = {vel_r2:.4f}")
    
    # --- Probe 3: Direction classification (8-way) ---
    print("  [Probe 3] Direction classification (accuracy)...")
    dir_feats = feat_tensor[:, :-1].reshape(N * (T-1), D)    # (N*19, 128)
    dir_targets = direction_classes.reshape(N * (T-1))        # (N*19,)
    dir_acc, _ = train_classification_probe(dir_feats, dir_targets)
    print(f"    Direction accuracy = {dir_acc:.4f} (chance = {1/8:.4f})")
    
    results[model_name] = {
        "pos_r2": pos_r2,
        "vel_r2": vel_r2,
        "dir_acc": dir_acc,
    }

print(f"\nAll probes trained. Models evaluated: {list(results.keys())}")

## Cell 5: Results Table

Formatted comparison table of all linear probe results.

How to read it:

- **Position R2:** How well does a linear function recover object location from frozen features? R2=0.95 means 95% of position variance explained.
- **Velocity R2:** Can we recover speed and direction of motion? Lower R2 expected since the encoder only sees single frames. Anything well above 0 (and above Random) means the encoder captures motion-relevant structure.
- **Direction Acc:** Can we classify which of 8 compass directions the object heads? Chance is 12.5%. Above 40% shows meaningful direction separation in feature space.

What to look for:
- Trained models should strongly outperform Random on all metrics.
- JEPA beating Pixel suggests latent-space prediction yields richer representations.
- Pixel matching JEPA means the simpler pixel objective already suffices.
- Random R2 should sit near 0; Random accuracy near 12.5%.

In [ ]:
# â”€â”€ Print results table â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
model_names = list(results.keys())

header = f"{'Metric':<20s}" + "".join(f"{name:>16s}" for name in model_names)
print("\n" + "=" * len(header))
print("LINEAR PROBE RESULTS")
print("=" * len(header))
print(header)
print("-" * len(header))

# Position RÂ²
row = f"{'Position RÂ²':<20s}"
for name in model_names:
    row += f"{results[name]['pos_r2']:>16.4f}"
print(row)

# Velocity RÂ²
row = f"{'Velocity RÂ²':<20s}"
for name in model_names:
    row += f"{results[name]['vel_r2']:>16.4f}"
print(row)

# Direction accuracy
row = f"{'Direction Acc':<20s}"
for name in model_names:
    row += f"{results[name]['dir_acc']:>16.4f}"
print(row)

# Chance-level reference
row = f"{'(chance level)':<20s}"
for name in model_names:
    row += f"{'0.1250':>16s}"
print(row)

print("=" * len(header))
print("\nNote: RÂ² = 1.0 is perfect; RÂ² = 0.0 means no better than predicting the mean.")
print("Direction chance level = 1/8 = 12.5% (8 compass directions).")

## Cell 6: Bar Chart

Three-panel grouped bar chart (position R2, velocity R2, direction accuracy) comparing all models.

Blue = Pixel, Purple = JEPA, Gray = Random. Each bar labeled with its numeric value. Direction panel includes a dashed red line at 12.5% (chance).

Saved to `paper/figures/fig_linear_probes.{pdf,png}` (12x4 in, 300 DPI, serif fonts).

The visual gap between trained (colored) and Random (gray) bars is the key result. A large gap proves the encoder learned structure beyond architectural inductive bias.

In [ ]:
# â”€â”€ Publication figure â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
matplotlib.rcParams.update({
    'font.family': 'serif',
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
})

# Colors for each model
COLOR_MAP = {
    "Pixel":  "#1565C0",
    "JEPA":   "#7B1FA2",
    "Random": "#9E9E9E",
}

model_names = list(results.keys())
colors = [COLOR_MAP.get(n, "#333333") for n in model_names]

fig, axes = plt.subplots(1, 3, figsize=(12, 4), dpi=300)

# Panel 1: Position RÂ²
ax = axes[0]
vals = [results[n]["pos_r2"] for n in model_names]
bars = ax.bar(model_names, vals, color=colors, edgecolor="black", linewidth=0.5)
ax.set_ylabel("RÂ² Score")
ax.set_title("Position Prediction")
ax.set_ylim(min(0, min(vals) - 0.05), 1.05)
ax.axhline(y=0, color="black", linewidth=0.5, linestyle="-")
for bar, v in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f"{v:.3f}", ha="center", va="bottom", fontsize=9)

# Panel 2: Velocity RÂ²
ax = axes[1]
vals = [results[n]["vel_r2"] for n in model_names]
bars = ax.bar(model_names, vals, color=colors, edgecolor="black", linewidth=0.5)
ax.set_ylabel("RÂ² Score")
ax.set_title("Velocity Prediction")
ax.set_ylim(min(0, min(vals) - 0.05), 1.05)
ax.axhline(y=0, color="black", linewidth=0.5, linestyle="-")
for bar, v in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f"{v:.3f}", ha="center", va="bottom", fontsize=9)

# Panel 3: Direction accuracy
ax = axes[2]
vals = [results[n]["dir_acc"] for n in model_names]
bars = ax.bar(model_names, vals, color=colors, edgecolor="black", linewidth=0.5)
ax.set_ylabel("Accuracy")
ax.set_title("Direction Classification")
ax.set_ylim(0, 1.05)
ax.axhline(y=1/8, color="red", linewidth=1, linestyle="--", label="Chance (12.5%)")
ax.legend(loc="upper right", fontsize=8)
for bar, v in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f"{v:.3f}", ha="center", va="bottom", fontsize=9)

fig.suptitle("Linear Probe Evaluation of Frozen Encoder Representations", fontsize=14, y=1.02)
plt.tight_layout()

# Save
fig.savefig(str(FIGURES_DIR / "fig_linear_probes.pdf"), bbox_inches="tight", dpi=300)
fig.savefig(str(FIGURES_DIR / "fig_linear_probes.png"), bbox_inches="tight", dpi=300)
print(f"Saved: {FIGURES_DIR / 'fig_linear_probes.pdf'}")
print(f"Saved: {FIGURES_DIR / 'fig_linear_probes.png'}")

plt.show()

print("\n--- Caption ---")
print("Figure: Linear probe evaluation. A frozen encoder produces latent features")
print("z in R^128. Simple linear classifiers trained on these features predict object")
print("position (RÂ²), velocity (RÂ²), and direction (accuracy). Higher scores indicate")
print("richer latent representations. Random baseline uses untrained weights.")

## Cell 7: t-SNE Visualization

Qualitative look at latent space structure. The linear probes give numbers; this plot lets you see the organization.

- **Input:** 500 samples at t=5, each a 128-dim feature vector
- **Coloring:** Motion direction class (8 compass directions), distinct hues
- **Model:** Best available trained model (JEPA if checkpoint exists, else Pixel)
- **Params:** perplexity=30, max_iter=1000

Saved to `paper/figures/fig_tsne_latent.{pdf,png}`.

How to interpret:
- Clear same-color clusters: the encoder groups samples by motion direction. Strong evidence of meaningful representation learning.
- Partial overlap with visible grouping: direction info present but with some ambiguity. Still a positive result.
- Fully mixed colors: the encoder does not organize by direction. Direction info may still exist (the linear probe found it) but is not the dominant organizing principle.

t-SNE is sensitive to perplexity and non-deterministic across runs. Don't over-interpret fine structural details. Focus on whether same-colored points cluster together.

In [ ]:
from sklearn.manifold import TSNE

# â”€â”€ t-SNE of latent space â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
N_TSNE = 500
T_TSNE = 5  # timestep to visualize

# Use the best trained model available (prefer Pixel, which always exists)
tsne_model_name = "Pixel"
if "JEPA" in features:
    tsne_model_name = "JEPA"

tsne_feats = features[tsne_model_name][:N_TSNE, T_TSNE].numpy()  # (500, 128)
# Direction at t=5 (velocity from t=5 to t=6, so use direction_classes[:, 5])
tsne_labels = direction_classes[:N_TSNE, T_TSNE].numpy()          # (500,)

print(f"Running t-SNE on {tsne_model_name} features at t={T_TSNE}...")
print(f"  Feature shape: {tsne_feats.shape}")
print(f"  Label shape:   {tsne_labels.shape}")

tsne = TSNE(n_components=2, perplexity=30, random_state=SEED, max_iter=1000)
tsne_2d = tsne.fit_transform(tsne_feats)  # (500, 2)

# â”€â”€ Plot â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
direction_names = ['E', 'NE', 'N', 'NW', 'W', 'SW', 'S', 'SE']
direction_colors = plt.cm.hsv(np.linspace(0, 1, 9))[:8]  # 8 distinct colors

fig, ax = plt.subplots(1, 1, figsize=(8, 7), dpi=300)

for cls_idx in range(8):
    mask = tsne_labels == cls_idx
    if mask.sum() == 0:
        continue
    ax.scatter(tsne_2d[mask, 0], tsne_2d[mask, 1],
               c=[direction_colors[cls_idx]], label=direction_names[cls_idx],
               alpha=0.6, s=20, edgecolors='none')

ax.set_xlabel("t-SNE Dim 1")
ax.set_ylabel("t-SNE Dim 2")
ax.set_title(f"t-SNE of {tsne_model_name} Latent Space (t={T_TSNE}, colored by direction)")
ax.legend(title="Direction", loc="best", fontsize=8, markerscale=1.5)
ax.set_aspect('equal', adjustable='datalim')

plt.tight_layout()
fig.savefig(str(FIGURES_DIR / "fig_tsne_latent.pdf"), bbox_inches="tight", dpi=300)
fig.savefig(str(FIGURES_DIR / "fig_tsne_latent.png"), bbox_inches="tight", dpi=300)
print(f"Saved: {FIGURES_DIR / 'fig_tsne_latent.pdf'}")
print(f"Saved: {FIGURES_DIR / 'fig_tsne_latent.png'}")
plt.show()

## Cell 8: Save Results

Saves all probe results to `experiments/analysis/linear_probe_results.npz`.

Contents: per-model metrics (`{ModelName}_pos_r2`, `{ModelName}_vel_r2`, `{ModelName}_dir_acc`), plus metadata (`model_names`, `n_sequences` = 2000, `n_probe_epochs` = 50).

The saved file feeds into LaTeX table generation and cross-experiment comparisons. Figures from Cells 6-7 go directly into `paper/figures/`.

In [ ]:
# â”€â”€ Save results â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
save_dir = ANALYSIS_DIR
save_path = save_dir / "linear_probe_results.npz"

save_dict = {}
for model_name, res in results.items():
    for metric_name, value in res.items():
        key = f"{model_name}_{metric_name}"
        save_dict[key] = np.array(value)

# Also save the model names for reference
save_dict["model_names"] = np.array(list(results.keys()))
save_dict["n_sequences"] = np.array(N_SEQ)
save_dict["n_probe_epochs"] = np.array(50)

np.savez(str(save_path), **save_dict)
print(f"Results saved to: {save_path}")
print(f"\nContents:")
for k, v in save_dict.items():
    print(f"  {k}: {v}")

print("\n" + "=" * 60)
print("LINEAR PROBE EVALUATION COMPLETE")
print("=" * 60)
print(f"  Models evaluated:  {list(results.keys())}")
print(f"  Sequences used:    {N_SEQ}")
print(f"  Figures saved to:  {FIGURES_DIR}")
print(f"  Results saved to:  {save_path}")